
# What is Databricks SQL's read_files function?
The `read_files` function lets you directly query and ingest files (like `CSV`, `JSON`, `Parquet`, etc.) from cloud storage or Unity Catalog using SQL—no table setup needed.

It supports ad-hoc exploration and incremental ingestion, including with `STREAMING TABLES`, and automatically infers schema and handles directories or patterns.

*Note: Spark Declarative Pipelines use read_files to easily load file data into tables, powering both batch and streaming workflows.*

For more details, open [the read_files documentation](https://docs.databricks.com/aws/en/sql/language-manual/functions/read_files).

In [0]:
%run ./_resources/00-setup $reset_all_data=false

USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_data_ingestion`


data already existing. Run with reset_all_data=true to force a data cleanup for your local demo.


True

In [0]:
display(dbutils.fs.ls(volume_folder))

path,name,size,modificationTime
dbfs:/Volumes/main/dbdemos_data_ingestion/raw_data/_wait_rescued/,_wait_rescued/,0,1761073125245
dbfs:/Volumes/main/dbdemos_data_ingestion/raw_data/checkpoint/,checkpoint/,0,1761073125245
dbfs:/Volumes/main/dbdemos_data_ingestion/raw_data/csv_streaming_schema/,csv_streaming_schema/,0,1761073125245
dbfs:/Volumes/main/dbdemos_data_ingestion/raw_data/parquet_streaming_schema/,parquet_streaming_schema/,0,1761073125245
dbfs:/Volumes/main/dbdemos_data_ingestion/raw_data/read_files_streaming_schema/,read_files_streaming_schema/,0,1761073125245
dbfs:/Volumes/main/dbdemos_data_ingestion/raw_data/user_csv/,user_csv/,0,1761073125245
dbfs:/Volumes/main/dbdemos_data_ingestion/raw_data/user_csv_no_headers/,user_csv_no_headers/,0,1761073125245
dbfs:/Volumes/main/dbdemos_data_ingestion/raw_data/user_csv_pipe_delimited/,user_csv_pipe_delimited/,0,1761073125245
dbfs:/Volumes/main/dbdemos_data_ingestion/raw_data/user_json/,user_json/,0,1761073125245
dbfs:/Volumes/main/dbdemos_data_ingestion/raw_data/user_parquet/,user_parquet/,0,1761073125245



## 1. Basic Usage: Automatic Format Dectection

One of the key advantages of `read_files` is automatic format detection. Let's try to read many file formats in our demo data folder, and use read_files to detect the file format

In [0]:
display(spark.sql(f"SELECT * FROM read_files('{volume_folder}/user_json') LIMIT 5"))

address,age_group,creation_date,email,firstname,gender,id,lastname,new_column,_rescued_data
241 Veronica Islands Suite 936 Lake Chelsea HI 69280,5.0,07-01-2025 16:51:36,fordsteven@moore.com,Antonio,1.0,10665,Johnson,null,null
Unit 3560 Box 2007 DPO AP 41729,3.0,07-10-2025 13:34:34,sanchezeric@williams-cohen.com,Shane,1.0,11334,West,null,null
37829 Stephen Path Andrewtown PW 67701,3.0,07-04-2025 10:51:16,alvarezkathleen@wright.com,Evan,1.0,2567,Martin,null,null
7021 Ricky Keys Port Kathryn MI 90090,3.0,07-10-2025 00:57:07,woodlance@stewart-mitchell.org,Rodney,0.0,230,Henderson,null,null
851 Sullivan Port Suite 759 Valdezside MT 73591,0.0,07-18-2025 11:22:38,clarsen@reyes.com,Lori,1.0,11771,Cardenas,null,null


In [0]:
display(spark.sql(f"SELECT * FROM read_files('{volume_folder}/user_csv') LIMIT 5"))

id,creation_date,firstname,lastname,email,address,gender,age_group,_rescued_data
11801,07-04-2025 22:51:37,Jacob,Lopez,stephaniewilson@morales-morrison.com,32945 Samuel Rapid Thompsonview VA 80359,1.0,2.0,null
5426,07-04-2025 06:37:00,Eric,Hernandez,oromero@stevenson.org,707 Matthew Trail Suite 774 Alexanderport NV 99538,1.0,8.0,null
4008,07-14-2025 11:50:38,Crystal,Chen,halecarol@pennington.com,0359 Richardson Views Suite 467 Mclaughlinstad AS 30330,0.0,7.0,null
10283,07-08-2025 14:35:03,Michael,Kelly,thomasfox@odom.com,178 Paul Shore Apt. 607 Tamaramouth OK 44693,0.0,7.0,null
3333,07-16-2025 06:53:36,Jennifer,Baldwin,james07@carroll.com,3278 Johnson Branch Apt. 132 South Meganbury TX 02578,0.0,4.0,null


In [0]:
display(spark.sql(f"SELECT * FROM read_files('{volume_folder}/user_parquet') LIMIT 5"))

id,creation_date,firstname,lastname,email,address,gender,age_group,_rescued_data
8257,07-17-2025 13:38:01,Kimberly,Parsons,jeffery23@jones-jackson.com,58910 Lopez Summit Angelaview CA 29633,0.0,4.0,null
4977,07-13-2025 04:48:54,Ashley,Duncan,johnwalker@clayton.com,04412 Hernandez Pine Lake Johnmouth DC 03526,0.0,3.0,null
2544,07-04-2025 18:42:57,Gregory,Thompson,richardadam@smith.com,52130 Kristen Common Wigginsbury NY 35023,1.0,4.0,null
10959,07-06-2025 08:06:03,John,Wright,donnamarsh@brown.com,55316 Moran Wells Suite 362 North Claudiaside AS 45321,1.0,6.0,null
9828,07-12-2025 13:40:37,Amber,Weaver,eric69@mitchell.com,2489 Russo Shoals Apt. 570 Port Ralphberg WV 14515,1.0,9.0,null


In [0]:
display(spark.sql(f"SELECT year, month, COUNT(*) as records FROM read_files('{volume_folder}/user_parquet_partitioned') GROUP BY year, month"))

year,month,records
2024,5,8361
2024,10,8376
2024,2,8408
2024,1,8372
2024,11,8431
2024,4,8273
2024,8,8449
2024,7,8325
2024,12,8306
2024,3,8274



`read_files` also supports powerful glob patterns for selective file reading. You can select the specific format you want to read.

In [0]:
display(spark.sql(f"SELECT 'JSON Files' as source, * FROM read_files('{volume_folder}/*json*') LIMIT 3"))

source,address,age_group,creation_date,email,firstname,gender,id,lastname,new_column,_rescued_data
JSON Files,241 Veronica Islands Suite 936 Lake Chelsea HI 69280,5.0,07-01-2025 16:51:36,fordsteven@moore.com,Antonio,1.0,10665,Johnson,null,null
JSON Files,Unit 3560 Box 2007 DPO AP 41729,3.0,07-10-2025 13:34:34,sanchezeric@williams-cohen.com,Shane,1.0,11334,West,null,null
JSON Files,37829 Stephen Path Andrewtown PW 67701,3.0,07-04-2025 10:51:16,alvarezkathleen@wright.com,Evan,1.0,2567,Martin,null,null



## 2. Schema Inference

Different formats have different schema inference capabilities and performance.

We can also use schema hints to override the schema inferrence.

In [0]:
json_schema = spark.sql(f"SELECT * FROM read_files('{volume_folder}/user_json') LIMIT 0").schema
print(json_schema.treeString())

root
 |-- address: string (nullable = true)
 |-- age_group: double (nullable = true)
 |-- creation_date: string (nullable = true)
 |-- email: string (nullable = true)
 |-- firstname: string (nullable = true)
 |-- gender: double (nullable = true)
 |-- id: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- new_column: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
csv_schema = spark.sql(f"SELECT * FROM read_files('{volume_folder}/user_csv') LIMIT 0").schema  
print(csv_schema.treeString())

root
 |-- id: long (nullable = true)
 |-- creation_date: string (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- email: string (nullable = true)
 |-- address: string (nullable = true)
 |-- gender: double (nullable = true)
 |-- age_group: double (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
display(spark.sql(f"""
SELECT
  format,
  MAX(id_type) AS id_type,
  MAX(age_group_type) AS age_group_type,
  MAX(date_type) AS date_type
FROM (
SELECT 
  'JSON' as format,
  typeof(id) as id_type,
  typeof(age_group) as age_group_type,
  typeof(creation_date) as date_type
FROM read_files('{volume_folder}/user_json')
UNION ALL
SELECT 
  'CSV' as format,
  typeof(id) as id_type, 
  typeof(age_group) as age_group_type,
  typeof(creation_date) as date_type
FROM read_files('{volume_folder}/user_csv')
UNION ALL
SELECT 
  'Parquet' as format,
  typeof(id) as id_type,
  typeof(age_group) as age_group_type, 
  typeof(creation_date) as date_type
FROM read_files('{volume_folder}/user_parquet')
) type_comparision
GROUP BY format
"""))

format,id_type,age_group_type,date_type
CSV,bigint,double,string
JSON,string,double,string
Parquet,bigint,double,string


In [0]:
display(spark.sql(f"""
SELECT 
  id,
  typeof(id) as id_type_after_hint,
  age_group,
  typeof(age_group) as age_group_type_after_hint
FROM read_files(
  '{volume_folder}/user_json',
  schemaHints => 'id bigint, age_group string'
) LIMIT 5
"""))

id,id_type_after_hint,age_group,age_group_type_after_hint
10665,bigint,5.0,string
11334,bigint,3.0,string
2567,bigint,3.0,string
230,bigint,3.0,string
11771,bigint,0.0,string



## 3. Format-Specific Features

There are some particular options that are specific to each format with `read_files`

In [0]:
display(spark.sql(f"SELECT * FROM read_files('{volume_folder}/user_csv_no_headers', format => 'csv', header => 'false') LIMIT 5"))

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_rescued_data
0,07-14-2025 11:59:14,Cynthia,Chen,sarahross@anderson.com,5026 Erin Plains Lake Amber SC 00544,1.0,9.0,null
1,07-20-2025 20:42:38,Sarah,Conrad,ellencarpenter@hall.biz,735 Stevens Square Apt. 776 Vargasstad NV 86971,0.0,2.0,null
2,07-09-2025 12:18:10,Raymond,Ross,ushepard@patterson.info,5292 Owen Lake Suite 970 Ortegaside LA 49028,1.0,4.0,null
3,07-01-2025 19:50:36,Benjamin,Williams,eknight@white.com,PSC 0178 Box 5286 APO AP 11815,0.0,2.0,null
4,07-09-2025 12:25:38,Raymond,Ross,thomas52@white.com,1472 Megan Trail Suite 281 East Michelle NY 93965,0.0,0.0,null


In [0]:
display(spark.sql(f"""
SELECT * FROM read_files(
  '{volume_folder}/user_csv_no_headers',
  format => 'csv',
  schema => 'id bigint, creation_date string, firstname string, lastname string, email string, address string, gender double, age_group double'
) LIMIT 5
"""))

id,creation_date,firstname,lastname,email,address,gender,age_group
0,07-14-2025 11:59:14,Cynthia,Chen,sarahross@anderson.com,5026 Erin Plains Lake Amber SC 00544,1.0,9.0
1,07-20-2025 20:42:38,Sarah,Conrad,ellencarpenter@hall.biz,735 Stevens Square Apt. 776 Vargasstad NV 86971,0.0,2.0
2,07-09-2025 12:18:10,Raymond,Ross,ushepard@patterson.info,5292 Owen Lake Suite 970 Ortegaside LA 49028,1.0,4.0
3,07-01-2025 19:50:36,Benjamin,Williams,eknight@white.com,PSC 0178 Box 5286 APO AP 11815,0.0,2.0
4,07-09-2025 12:25:38,Raymond,Ross,thomas52@white.com,1472 Megan Trail Suite 281 East Michelle NY 93965,0.0,0.0


In [0]:
display(spark.sql(f"""
SELECT * FROM read_files(
  '{volume_folder}/user_csv_pipe_delimited',
  format => 'csv',
  sep => '|'  
) LIMIT 5
"""))

id,creation_date,firstname,lastname,email,address,gender,age_group,_rescued_data
0,07-14-2025 11:59:24,Cynthia,Chen,sarahross@anderson.com,5026 Erin Plains Lake Amber SC 00544,1.0,9.0,null
1,07-20-2025 20:42:52,Sarah,Conrad,ellencarpenter@hall.biz,735 Stevens Square Apt. 776 Vargasstad NV 86971,0.0,2.0,null
2,07-09-2025 12:18:16,Raymond,Ross,ushepard@patterson.info,5292 Owen Lake Suite 970 Ortegaside LA 49028,1.0,4.0,null
3,07-01-2025 19:50:37,Benjamin,Williams,eknight@white.com,PSC 0178 Box 5286 APO AP 11815,0.0,2.0,null
4,07-09-2025 12:25:44,Raymond,Ross,thomas52@white.com,1472 Megan Trail Suite 281 East Michelle NY 93965,0.0,0.0,null


In [0]:
display(spark.sql(f"""
SELECT 
  firstname,
  lastname,
  id,
  typeof(id) as id_inferred_type,
  age_group,
  typeof(age_group) as age_group_inferred_type
FROM read_files(
  '{volume_folder}/user_json',
  inferColumnTypes => true
) LIMIT 5  
"""))

firstname,lastname,id,id_inferred_type,age_group,age_group_inferred_type
Antonio,Johnson,10665,string,5.0,double
Shane,West,11334,string,3.0,double
Evan,Martin,2567,string,3.0,double
Rodney,Henderson,230,string,3.0,double
Lori,Cardenas,11771,string,0.0,double


In [0]:
# Demonstrate column pruning (Parquet's key advantage)
print("⚡ Parquet Column Pruning Demo:")
import time

# Read all columns
start_time = time.time()
all_cols_count = spark.sql(f"SELECT * FROM read_files('{volume_folder}/user_parquet')").count()
all_cols_time = time.time() - start_time

# Read only specific columns  
start_time = time.time()
select_cols_count = spark.sql(f"SELECT id, firstname FROM read_files('{volume_folder}/user_parquet')").count()
select_cols_time = time.time() - start_time

print(f"📊 All columns: {all_cols_count:,} records in {all_cols_time:.2f}s")
print(f"📊 2 columns: {select_cols_count:,} records in {select_cols_time:.2f}s") 
print(f"⚡ Column pruning speedup: {all_cols_time/select_cols_time:.1f}x faster")

⚡ Parquet Column Pruning Demo:
📊 All columns: 100,000 records in 0.59s
📊 2 columns: 100,000 records in 0.39s
⚡ Column pruning speedup: 1.5x faster


## 4. Streaming Usage

`read_files` can be used in streaming tables to ingest files into Delta Lake. `read_files` leverages Auto Loader when used in a streaming table query.

To do so, simply add the `STREAM` keyword to your SQL queries:

In [0]:
# Create a streaming view
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW streaming_json_users AS
SELECT 
  *,
  current_timestamp() as processing_time
FROM STREAM read_files(
  '{volume_folder}/user_json',
  maxFilesPerTrigger => 5,
  schemaLocation => '{volume_folder}/read_files_streaming_schema'
)
""")

display(spark.sql("SELECT COUNT(*) as total_records FROM streaming_json_users"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW streaming_csv_users AS  
SELECT 
  *,
  'CSV' as source_format,
  current_timestamp() as processing_time
FROM STREAM read_files(
  '{volume_folder}/user_csv',
  schemaLocation => '{volume_folder}/csv_streaming_schema'
)
""")

spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW streaming_parquet_users AS
SELECT 
  *,
  'PARQUET' as source_format, 
  current_timestamp() as processing_time
FROM STREAM read_files(
  '{volume_folder}/user_parquet',
  schemaLocation => '{volume_folder}/parquet_streaming_schema'
)
""")

display(spark.sql("""
SELECT 'CSV' as format, COUNT(*) as records FROM streaming_csv_users
UNION ALL  
SELECT 'PARQUET' as format, COUNT(*) as records FROM streaming_parquet_users
"""))


## 5. read_files vs Auto Loader

We have covered some basic features of `read_files`. However, there might be some questions about when to use `read_files` and when to use Auto Loader.

We have some comparison and decision matrix that could help you decide when to leverage the power of `read_files` and Auto Loader.


| Capability | read_files | Auto Loader |
|-----------|------------|-------------|
| Language | SQL | Python  |
| Ad-hoc queries | ✅ Perfect | Incremental Streaming focus  |
| Batch processing | ✅ Excellent | Incremental Streaming focus |
| Multi-format API | ✅ Unified API | Need to declare format |
| Streaming performance | Optimized for you | Mode advanced options for more control |
| Schema evolution | ⚠️ Manual | ✅ Automatic |
| Setup complexity | ✅ Zero setup | Pythonic config |
| File notifications | ❌ No | ✅ Cloud notifications |


## Conclusion

We have seen what the capabilities of Databricks SQL's `read_files` are, and now you can apply it in your projects.

Open the [02-Auto-loader-schema-evolution-Ingestion]($./02-Auto-loader-schema-evolution-Ingestion) Notebook to explore the Auto Loader options!

In [0]:
DBDemos.stop_all_streams()

Stopping 2 streams
All stream stopped 
